In [ ]:
#%pip install tqdm
#%pip install ultralytics
#%pip install pandas
#%pip install scikit-learn

from ultralytics import YOLO
import pandas as pd
import re
import shutil
from sklearn.model_selection import train_test_split



/home/daniel/Documents/deepLearning/Project2_TrOCR/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
/home/daniel/Documents/deepLearning/Project2_TrOCR/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.


In [2]:
from pathlib import Path
import os

# Run once
os.chdir(Path.cwd().parent)


## Extract crops

In [ ]:
class yolo:
    def __init__(self, model_path):
        self.model = YOLO(model_path)

    def predict_and_save(self, img_path, save_path):
        # Input folder path, run inference over all images in the folder, and save crops
        self.model.predict(img_path,
                        imgsz=896,
                        device=0,
                        conf=0.6,
                        save_crop = True,
                        project = save_path
                        ) 
        
        

In [ ]:
yolo_model = yolo("models/customYOLO26N.pt")

yolo_model.predict_and_save("data/rawImages/test", "/home/daniel/Documents/deepLearning/Project2_TrOCR/data/OCRcrops/test")

yolo_model.predict_and_save("data/rawImages/train", "/home/daniel/Documents/deepLearning/Project2_TrOCR/data/OCRcrops/train")

yolo_model.predict_and_save("data/rawImages/val", "/home/daniel/Documents/deepLearning/Project2_TrOCR/data/OCRcrops/val")


image 1/9 /home/daniel/Documents/deepLearning/Project2_TrOCR/data/rawImages/test/IMG_0846_jpeg.rf.9a793b4384afca75ab9a22146e6fcebd.jpg: 896x672 10 rotors, 17.3ms
image 2/9 /home/daniel/Documents/deepLearning/Project2_TrOCR/data/rawImages/test/IMG_0848_jpeg.rf.d6b99bc6ba421916811371f16dece3fa.jpg: 896x672 39 rotors, 18.0ms
image 3/9 /home/daniel/Documents/deepLearning/Project2_TrOCR/data/rawImages/test/IMG_0849_jpeg.rf.fa71aa8415322bf4f7c5fcbaf9302d58.jpg: 896x672 24 rotors, 18.2ms
image 4/9 /home/daniel/Documents/deepLearning/Project2_TrOCR/data/rawImages/test/IMG_0852_jpeg.rf.f8b79984537a484a1539b67b61ca18a2.jpg: 896x672 19 rotors, 15.4ms
image 5/9 /home/daniel/Documents/deepLearning/Project2_TrOCR/data/rawImages/test/IMG_0853_jpeg.rf.6826179f74082027ed6959aa9273960b.jpg: 896x672 11 rotors, 18.3ms
image 6/9 /home/daniel/Documents/deepLearning/Project2_TrOCR/data/rawImages/test/IMG_0855_jpeg.rf.80474a7b18d63c1f595eeede6e4a8f8d.jpg: 896x672 35 rotors, 18.1ms
image 7/9 /home/daniel/Docu

# Reference baselineModel.ipynb for next steps

## Re split CSV's to decrease bias

In [ ]:
test = pd.read_csv("data/OCRcrops/test/tradOutputTest/annotatedTest.csv").dropna(subset=["annotated_transcription"])

train = pd.read_csv("data/OCRcrops/train/tradOutputTrain/annotatedTrain.csv").dropna(subset=["annotated_transcription"])

val = pd.read_csv("data/OCRcrops/val/tradOutputVal/annotatedVal.csv").dropna(subset=["annotated_transcription"])

In [10]:
print(test.shape)

print(train.shape)

print(val.shape)

(176, 9)
(721, 9)
(173, 9)


In [ ]:
# Load both files and combine
full_df = pd.concat([train, val, test], ignore_index=True)

print(f"Full dataset size: {len(full_df)}")
print(f"Unique part numbers in full dataset: {full_df['annotated_transcription'].nunique()}")


Full dataset size: 1070
Unique part numbers in full dataset: 270


In [ ]:
# Check for biases
full_df = full_df[full_df["annotated_transcription"].str.len() != 6]


# potential bias: suffix, prefix, digit length, label length
full_df["num_digits"]   = full_df["annotated_transcription"].str.extract(r'^(\d+)')[0].str.len()
full_df["suffix"]       = full_df["annotated_transcription"].str.extract(r'([A-Za-z]+)$')[0].fillna("none")
full_df["first_digit"]  = full_df["annotated_transcription"].str[0]  
full_df["label_length"] = full_df["annotated_transcription"].str.len()

print(full_df["suffix"].value_counts())
print(full_df["num_digits"].value_counts())
print(full_df["label_length"].value_counts())



suffix
none    668
DL      401
Name: count, dtype: int64
num_digits
5    1027
4      42
Name: count, dtype: int64
label_length
5    626
7    401
4     42
Name: count, dtype: int64


In [ ]:
# Combines suffix + digit length so both are balanced across splits
full_df["strat_key"] = full_df["suffix"] + "_" + full_df["num_digits"].astype(str)
print(full_df["strat_key"].value_counts())


# Split on strat, create temp for val/test split
train_df, temp_df = train_test_split(
    full_df, test_size=0.2, stratify=full_df["strat_key"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["strat_key"], random_state=42
)

# check biases
for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"\n{name} ({len(split)} samples)")
    print(split["suffix"].value_counts(normalize=True).round(2))
    print(split["num_digits"].value_counts(normalize=True).round(2))

train_ids = set(train_df["id"])
val_ids   = set(val_df["id"])
test_ids  = set(test_df["id"])
print(f"\nTrain/val overlap  : {len(train_ids & val_ids)}")
print(f"Train/test overlap : {len(train_ids & test_ids)}")
print(f"Val/test overlap   : {len(val_ids & test_ids)}")

strat_key
none_5    626
DL_5      401
none_4     42
Name: count, dtype: int64

train (855 samples)
suffix
none    0.62
DL      0.38
Name: proportion, dtype: float64
num_digits
5    0.96
4    0.04
Name: proportion, dtype: float64

val (107 samples)
suffix
none    0.63
DL      0.37
Name: proportion, dtype: float64
num_digits
5    0.96
4    0.04
Name: proportion, dtype: float64

test (107 samples)
suffix
none    0.63
DL      0.37
Name: proportion, dtype: float64
num_digits
5    0.95
4    0.05
Name: proportion, dtype: float64

Train/val overlap  : 0
Train/test overlap : 0
Val/test overlap   : 0


In [ ]:
print(train_df.shape)

print(val_df.shape)

print(test_df.shape)

(855, 14)
(107, 14)
(107, 14)


In [ ]:
train_images = set(train_df["image"])
val_images   = set(val_df["image"])
test_images  = set(test_df["image"])

print(f"Train/val overlap  : {len(train_images & val_images)}")
print(f"Train/test overlap : {len(train_images & test_images)}")
print(f"Val/test overlap   : {len(val_images   & test_images)}")

Train/val overlap  : 0
Train/test overlap : 0
Val/test overlap   : 0


In [ ]:
# Save the new splits
train_df.to_csv("annotatedTrainV2.csv", index=False)
val_df.to_csv("annotatedValV2.csv",   index=False)
test_df.to_csv("annotatedTestV2.csv",  index=False)

## Split image files

In [ ]:
train = pd.read_csv("data/OCRcropsv2/annotatedTrainV2.csv")
val = pd.read_csv("data/OCRcropsv2/annotatedValV2.csv")
test = pd.read_csv("data/OCRcropsv2/annotatedTestV2.csv")

# Extract just the filename from the image path
for df in [train, val, test]:
    df["image"] = df["image"].str.split("?d=", regex=False).str[1].apply(os.path.basename)

# Create destination folder
train_dir = "data/OCRcropsv2/train"
val_dir = "data/OCRcropsv2/val"
test_dir = "data/OCRcropsv2/test"

os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)


dfs = [train, val, test]

dirs = [train_dir, val_dir, test_dir]

base = "data/OCRcropsv2/"

# Copy images to respective folders
for x, y in zip(dfs, dirs):
    for img in x["image"]:
        path = os.path.join(base, img)
        if os.path.exists(path):
            shutil.copy(path, y)
        else:
            print(f"Image not found: {img}")

In [ ]:
# Update image paths in CSVs to reflect new folder structure
train = pd.read_csv("data/OCRcropsv2/train/tradOutputTrain/annotatedTrainV2.csv")
val = pd.read_csv("data/OCRcropsv2/val/tradOutputVal/annotatedValV2.csv")
test = pd.read_csv("data/OCRcropsv2/test/tradOutputTest/annotatedTestV2.csv")

train["image"] = train["image"].str.replace("rawCropTest|rawCropVal", "rawCropTrain", regex=True)
val["image"] = val["image"].str.replace("rawCropTrain|rawCropTest", "rawCropVal", regex=True)
test["image"] = test["image"].str.replace("rawCropTrain|rawCropVal", "rawCropTest", regex=True)

train.to_csv("annotatedTrainV3.csv", index=False)
val.to_csv("annotatedValV3.csv",   index=False)
test.to_csv("annotatedTestV3.csv",  index=False)



In [ ]:
# Final check for biases in new splits
test = pd.read_csv("data/OCRcropsv3/test/tradOutputTest/annotatedTestV3.csv").dropna(subset=["annotated_transcription"])

train = pd.read_csv("data/OCRcropsv3/train/tradOutputTrain/annotatedTrainV3.csv").dropna(subset=["annotated_transcription"])

val = pd.read_csv("data/OCRcropsv3/val/tradOutputVal/annotatedValV3.csv").dropna(subset=["annotated_transcription"])

dfs = {"train": train, "test": test, "val": val}

for name, df in dfs.items():
    df["first_digit"] = df["annotated_transcription"].str[0]

    print(name)
    print(df["first_digit"].value_counts().sort_index() / len(df))


train
first_digit
3    0.200000
4    0.307602
5    0.302924
6    0.005848
7    0.183626
Name: count, dtype: float64
test
first_digit
3    0.214953
4    0.242991
5    0.345794
6    0.009346
7    0.186916
Name: count, dtype: float64
val
first_digit
3    0.186916
4    0.271028
5    0.308411
6    0.009346
7    0.224299
Name: count, dtype: float64


# Reference Eval.ipynb for final evaluation of baseline model on the finalized data split